# 02 — Modeling: Baselines and XGBoost

Evaluated on the last 28 days of data (2016-04-25 to 2016-05-22) as a held-out test set. All models forecast this fixed 28-day horizon from a single origin (the day before holdout start), using only training history.

This notebook covers:
1. Baseline models (naive, moving average, seasonal naive)
2. XGBoost with engineered features, tuned via rolling-origin cross-validation
3. Prophet on the top 10 SKUs by volume

In [1]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from metrics import rmse, mae, wape
from models import naive_weekly_forecast, moving_average_forecast, seasonal_naive_forecast

df = pd.read_parquet("../data/processed/modeling_table.parquet")
print(f"{len(df):,} rows | {df['date'].min().date()} to {df['date'].max().date()}")

371,400 rows | 2013-01-01 to 2016-05-22


## Holdout split

Last 28 days held out as the final test set; everything before that is training history.

In [2]:
dates = sorted(df["date"].unique())
HOLDOUT_DATES = dates[-28:]
HOLDOUT_START = HOLDOUT_DATES[0]
HORIZON = len(HOLDOUT_DATES)

print(f"Holdout: {pd.Timestamp(HOLDOUT_START).date()} to {pd.Timestamp(HOLDOUT_DATES[-1]).date()} ({HORIZON} days)")
print(f"Training: {df['date'].min().date()} to {(pd.Timestamp(HOLDOUT_START) - pd.Timedelta(days=1)).date()}")

Holdout: 2016-04-25 to 2016-05-22 (28 days)
Training: 2013-01-01 to 2016-04-24


## Baseline models

Three baselines, each forecasting the full 28-day holdout from the training-only origin:

- **Naive (t-7)**: repeats the last observed week's day-of-week pattern across all 4 weeks of the horizon.
- **28-day moving average**: flat forecast at the trailing 28-day mean.
- **Seasonal naive (t-364)**: repeats the same 28 days from a year earlier; falls back to a 28-day tile for any series with under a year of history (none in this scope, since the shortest series covers 3.4 years).

In [3]:
BASELINE_MODELS = {
    "Naive (t-7)": naive_weekly_forecast,
    "28-day Moving Average": moving_average_forecast,
    "Seasonal Naive (t-364)": seasonal_naive_forecast,
}

def evaluate_baselines(df, holdout_start, horizon=28):
    """Run each baseline per store-item series and pool predictions for metric computation."""
    preds = {name: {"y_true": [], "y_pred": []} for name in BASELINE_MODELS}
    for (store, item), g in df.groupby(["store_id", "item_id"]):
        g = g.sort_values("date")
        train = g.loc[g["date"] < holdout_start, "sales"].values
        actual = g.loc[g["date"] >= holdout_start, "sales"].values
        for name, fn in BASELINE_MODELS.items():
            pred = fn(train, horizon=horizon)
            preds[name]["y_true"].append(actual)
            preds[name]["y_pred"].append(pred)
    return preds

baseline_preds = evaluate_baselines(df, HOLDOUT_START, HORIZON)

rows = []
for name, d in baseline_preds.items():
    y_true = np.concatenate(d["y_true"])
    y_pred = np.concatenate(d["y_pred"])
    rows.append({"model": name, "RMSE": rmse(y_true, y_pred), "MAE": mae(y_true, y_pred), "WAPE": wape(y_true, y_pred)})

baseline_results = pd.DataFrame(rows).set_index("model").round(3)
baseline_results

,RMSE,MAE,WAPE
model,,,
Naive (t-7),6.916,4.169,0.532
28-day Moving Average,6.085,3.705,0.473
Seasonal Naive (t-364),8.914,5.400,0.689


In [4]:
baseline_results.to_csv("../data/processed/baseline_results.csv")

md_table = baseline_results.to_markdown()
with open("../data/processed/baseline_results.md", "w") as f:
    f.write(md_table)
print(md_table)

| model                  |   RMSE |   MAE |   WAPE |
|:-----------------------|-------:|------:|-------:|
| Naive (t-7)            |  6.916 | 4.169 |  0.532 |
| 28-day Moving Average  |  6.085 | 3.705 |  0.473 |
| Seasonal Naive (t-364) |  8.914 | 5.4   |  0.689 |


**Takeaway:** The 28-day moving average is the strongest baseline (WAPE 0.473), beating both the weekly naive (0.532) and the yearly seasonal naive (0.689). The moving average wins because it smooths out day-to-day noise while still tracking the current demand level; the weekly naive is noisier because it copies a single week's random fluctuations; the yearly seasonal naive is worst because FOODS_3 demand here is driven far more by weekly cadence than by year-over-year seasonality, and a 364-day-old snapshot is a stale estimate of the current level. **The 28-day moving average is the baseline to beat in Phase 4.**

## XGBoost: feature engineering

Features (`src/features.py`): lags 7/14/28, rolling mean/std (7, 28) computed on prior-day sales only, day-of-week, month, event flag, SNAP, price, price-change flag, and label-encoded store/item id.

In [5]:
from features import build_training_table, iterative_forecast, FEATURE_COLS
from models import train_xgboost

train_raw = df[df["date"] < HOLDOUT_START]
train_table, store_map, item_map = build_training_table(train_raw)
print(f"Training table: {train_table.shape[0]:,} rows (first {28} days of each series dropped for lag history)")
print(f"Features: {FEATURE_COLS}")

Training table: 354,600 rows (first 28 days of each series dropped for lag history)
Features: ['lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_std_7', 'roll_mean_28', 'roll_std_28', 'wday', 'month', 'is_event', 'snap', 'sell_price', 'price_change_flag', 'store_enc', 'item_enc']


## Rolling-origin cross-validation

3 folds, each a 28-day horizon immediately preceding the previous one, walking backward from the final holdout. A light grid over `max_depth`, `learning_rate`, `n_estimators` (12 combos) is scored by mean CV WAPE across folds — kept small per the project's tuning budget.

In [6]:
import itertools

EXOG_COLS = ["store_id", "item_id", "date", "wday", "month", "is_event", "snap", "sell_price", "price_change_flag"]

def run_fold(fold_start, params):
    fold_idx = dates.index(fold_start)
    fold_dates = dates[fold_idx:fold_idx + HORIZON]
    fold_train_raw = df[df["date"] < fold_start]
    fold_table, f_store_map, f_item_map = build_training_table(fold_train_raw)
    model = train_xgboost(fold_table, params)
    sales_wide = fold_train_raw.pivot(index="date", columns=["store_id", "item_id"], values="sales")
    exog_df = df.loc[df["date"].isin(fold_dates), EXOG_COLS]
    preds = iterative_forecast(model, sales_wide, exog_df, f_store_map, f_item_map, fold_dates)
    actual = df.loc[df["date"].isin(fold_dates), ["store_id", "item_id", "date", "sales"]]
    merged = preds.merge(actual, on=["store_id", "item_id", "date"])
    return wape(merged["sales"], merged["sales_pred"])

fold_starts = [dates[-HORIZON * (k + 2)] for k in range(3)]
param_grid = list(itertools.product([3, 5, 7], [0.05, 0.1], [100, 300]))
print(f"{len(param_grid)} hyperparameter combos x {len(fold_starts)} folds")

cv_rows = []
for max_depth, lr, n_est in param_grid:
    params = {"max_depth": max_depth, "learning_rate": lr, "n_estimators": n_est}
    fold_wapes = [run_fold(fs, params) for fs in fold_starts]
    cv_rows.append({**params, "mean_cv_wape": np.mean(fold_wapes)})

cv_results = pd.DataFrame(cv_rows).sort_values("mean_cv_wape").reset_index(drop=True)
cv_results.to_csv("../data/processed/xgb_cv_results.csv", index=False)
cv_results.round(4)

12 hyperparameter combos x 3 folds


,max_depth,learning_rate,n_estimators,mean_cv_wape
0,5,0.05,100,0.4724
1,5,0.10,100,0.4770
2,3,0.10,100,0.4777
3,5,0.05,300,0.4788
4,7,0.05,100,0.4791
5,3,0.05,300,0.4802
6,3,0.05,100,0.4811
7,7,0.10,100,0.4819
8,5,0.10,300,0.4822
9,3,0.10,300,0.4822


In [7]:
best_params = cv_results.iloc[0][["max_depth", "learning_rate", "n_estimators"]].to_dict()
best_params["max_depth"] = int(best_params["max_depth"])
best_params["n_estimators"] = int(best_params["n_estimators"])
print("Best hyperparameters (by mean CV WAPE):", best_params)

Best hyperparameters (by mean CV WAPE): {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 100}


**Takeaway:** `max_depth=5, learning_rate=0.05, n_estimators=100` gives the lowest mean CV WAPE (0.472). Deeper trees (7) and more estimators (300) both overfit slightly and score worse on held-out folds — a shallow, moderately-sized model generalizes best on this scope.

## Final XGBoost model: forecast the true holdout

Retrain on all data through the day before the true 28-day holdout, using the CV-selected hyperparameters, then forecast recursively.

In [8]:
final_model = train_xgboost(train_table, best_params)

sales_wide = train_raw.pivot(index="date", columns=["store_id", "item_id"], values="sales")
exog_holdout = df.loc[df["date"].isin(HOLDOUT_DATES), EXOG_COLS]
xgb_preds = iterative_forecast(final_model, sales_wide, exog_holdout, store_map, item_map, HOLDOUT_DATES)

actual_holdout = df.loc[df["date"].isin(HOLDOUT_DATES), ["store_id", "item_id", "date", "sales"]]
xgb_merged = xgb_preds.merge(actual_holdout, on=["store_id", "item_id", "date"])
xgb_merged.to_parquet("../data/processed/xgb_holdout_preds.parquet", index=False)

xgb_overall = {"model": "XGBoost", "RMSE": rmse(xgb_merged["sales"], xgb_merged["sales_pred"]),
               "MAE": mae(xgb_merged["sales"], xgb_merged["sales_pred"]),
               "WAPE": wape(xgb_merged["sales"], xgb_merged["sales_pred"])}
pd.DataFrame([xgb_overall]).round(3)

,model,RMSE,MAE,WAPE
0,XGBoost,5.465,3.429,0.437


**Takeaway:** XGBoost reaches **WAPE 0.437** on the true holdout, beating the best baseline (28-day moving average, WAPE 0.473) by about 7.6% relative — the lag/rolling/calendar/price features capture patterns the simple baselines can't.

## Prophet: top 10 SKUs by volume

Per-SKU Prophet models (weekly + yearly seasonality) fit independently on the 10 highest-volume store-item series, forecasting the same 28-day holdout. Compared against XGBoost and the best baseline restricted to the same 10 series, since Prophet isn't fit on the full 300-series scope.

In [9]:
import logging
from prophet import Prophet

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

top10 = (
    train_raw.groupby(["store_id", "item_id"])["sales"].sum()
    .sort_values(ascending=False).head(10).index.tolist()
)

prophet_rows = []
for store, item in top10:
    sub = train_raw.loc[
        (train_raw["store_id"] == store) & (train_raw["item_id"] == item), ["date", "sales"]
    ].rename(columns={"date": "ds", "sales": "y"})
    m = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
    m.fit(sub)
    fc = m.predict(pd.DataFrame({"ds": HOLDOUT_DATES}))
    pred = np.clip(fc["yhat"].values, 0, None)
    for d, p in zip(HOLDOUT_DATES, pred):
        prophet_rows.append({"store_id": store, "item_id": item, "date": d, "sales_pred": p})

prophet_df = pd.DataFrame(prophet_rows).merge(actual_holdout, on=["store_id", "item_id", "date"])
prophet_df.to_parquet("../data/processed/prophet_holdout_preds.parquet", index=False)

top10_df = pd.DataFrame(top10, columns=["store_id", "item_id"])
xgb_top10 = xgb_merged.merge(top10_df, on=["store_id", "item_id"])
ma_top10_rows = []
for store, item in top10:
    g = df[(df["store_id"] == store) & (df["item_id"] == item)].sort_values("date")
    tr = g.loc[g["date"] < HOLDOUT_START, "sales"].values
    act = g.loc[g["date"] >= HOLDOUT_START, "sales"].values
    pred = moving_average_forecast(tr, horizon=HORIZON)
    ma_top10_rows.append(pd.DataFrame({"sales": act, "sales_pred": pred}))
ma_top10 = pd.concat(ma_top10_rows, ignore_index=True)

top10_results = pd.DataFrame([
    {"model": "28-day Moving Average", "WAPE": wape(ma_top10["sales"], ma_top10["sales_pred"])},
    {"model": "XGBoost", "WAPE": wape(xgb_top10["sales"], xgb_top10["sales_pred"])},
    {"model": "Prophet", "WAPE": wape(prophet_df["sales"], prophet_df["sales_pred"])},
]).set_index("model").round(3)
top10_results

Importing plotly failed. Interactive plots will not work.


18:46:06 - cmdstanpy - INFO - Chain [1] start processing


18:46:06 - cmdstanpy - INFO - Chain [1] done processing


18:46:06 - cmdstanpy - INFO - Chain [1] start processing


18:46:06 - cmdstanpy - INFO - Chain [1] done processing


...

18:46:07 - cmdstanpy - INFO - Chain [1] start processing


18:46:07 - cmdstanpy - INFO - Chain [1] done processing


18:46:07 - cmdstanpy - INFO - Chain [1] start processing


18:46:07 - cmdstanpy - INFO - Chain [1] done processing


,WAPE
model,
28-day Moving Average,0.279
XGBoost,0.220
Prophet,0.244


**Takeaway:** On the top-10 highest-volume SKUs, XGBoost still wins (WAPE 0.220) over Prophet (0.244) and the moving-average baseline (0.279). Prophet's per-SKU yearly+weekly seasonal decomposition helps over the naive baseline, but it can't use cross-SKU signal (price, SNAP, events shared across series) the way a single XGBoost model trained on all 300 series can — so Prophet isn't adopted as the primary model, though its per-SKU decomposition could still be a useful diagnostic for a handful of very high-volume items.

## Final results table: model × metric × segment

All models evaluated on the same 28-day holdout, broken out by the stable/volatile segments from `01_eda.ipynb`. All models are compared against the best baseline (28-day moving average).

In [10]:
seg = pd.read_parquet("../data/processed/sku_segments.parquet")[["store_id", "item_id", "segment"]]

def segment_results(pred_df, model_name):
    d = pred_df.merge(seg, on=["store_id", "item_id"])
    rows = [{"model": model_name, "segment": "overall",
             "RMSE": rmse(d["sales"], d["sales_pred"]), "MAE": mae(d["sales"], d["sales_pred"]),
             "WAPE": wape(d["sales"], d["sales_pred"])}]
    for s in ["stable", "volatile"]:
        sub = d[d["segment"] == s]
        rows.append({"model": model_name, "segment": s,
                     "RMSE": rmse(sub["sales"], sub["sales_pred"]), "MAE": mae(sub["sales"], sub["sales_pred"]),
                     "WAPE": wape(sub["sales"], sub["sales_pred"])})
    return rows

all_rows = []
for name, fn in BASELINE_MODELS.items():
    d = baseline_preds[name]
    pred_long = []
    for (store, item), y_true, y_pred in zip(df.groupby(["store_id", "item_id"]).groups.keys(), d["y_true"], d["y_pred"]):
        pred_long.append(pd.DataFrame({"store_id": store, "item_id": item, "date": HOLDOUT_DATES,
                                        "sales": y_true, "sales_pred": y_pred}))
    all_rows += segment_results(pd.concat(pred_long, ignore_index=True), name)
all_rows += segment_results(xgb_merged, "XGBoost")

final_results = pd.DataFrame(all_rows)[["model", "segment", "RMSE", "MAE", "WAPE"]].round(3)
final_results.to_csv("../data/processed/final_results_by_segment.csv", index=False)
with open("../data/processed/final_results_by_segment.md", "w") as f:
    f.write(final_results.to_markdown(index=False))
final_results

,model,segment,RMSE,MAE,WAPE
0,Naive (t-7),overall,6.916,4.169,0.532
1,Naive (t-7),stable,7.147,4.833,0.445
2,Naive (t-7),volatile,6.677,3.505,0.727
3,28-day Moving Average,overall,6.085,3.705,0.473
4,28-day Moving Average,stable,6.582,4.290,0.395
5,28-day Moving Average,volatile,5.545,3.121,0.648
6,Seasonal Naive (t-364),overall,8.914,5.400,0.689
7,Seasonal Naive (t-364),stable,9.606,6.127,0.564
8,Seasonal Naive (t-364),volatile,8.163,4.672,0.970
9,XGBoost,overall,5.465,3.429,0.437


**Takeaway:** XGBoost beats the best baseline (28-day moving average) in every segment — overall WAPE 0.437 vs. 0.473 (~7.6% relative improvement), stable-segment WAPE 0.354 vs. 0.395, and volatile-segment WAPE 0.626 vs. 0.648. The gap is proportionally similar across segments, meaning XGBoost isn't just picking up easy wins on the stable half — it improves forecasting for volatile, intermittent SKUs too, though volatile-segment error remains roughly 1.8x higher than stable in absolute WAPE terms regardless of model. This gap is explored further, along with its inventory implications, in `03_evaluation.ipynb`.